In [16]:
from pathlib import Path
import json
import re
from datetime import datetime

In [17]:

BRONZE_FOLDER = Path("../../Data/bronze")
SILVER_FOLDER = Path("../../Data/silver")

SILVER_FOLDER.mkdir(parents=True, exist_ok=True)


In [18]:
def normalize_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

In [19]:
def remove_table_of_contents(text):
    start = text.lower().find("inhoudsopgave")
    end = text.lower().find("managementsamenvatting")

    if start != -1 and end != -1 and end > start:
        return text[:start] + text[end:]

    return text


In [20]:

def remove_pdf_artifacts(text):
    # Remove dotted table-of-content lines
    text = re.sub(r"\.{5,}", " ", text)

    # Remove standalone page numbers
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)

    # Remove figure/table lines
    text = re.sub(r"^\s*(Afbeelding|Figuur|Tabel)\s+\d+.*$", "", text, flags=re.MULTILINE | re.IGNORECASE)

    # Remove very long numeric garbage strings
    text = re.sub(r"\b[\d,%.\-]{15,}\b", " ", text)

    # Remove urls
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    return text

In [21]:
def fix_spacing(text):
    # Fix spaces before punctuation
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)

    # Normalize empty lines
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    # Merge broken lines inside paragraphs
    text = re.sub(r"(?<![.!?:])\n(?!\n)", " ", text)

    # Final whitespace cleanup
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [22]:
def clean_text(text):
    text = normalize_text(text)
    text = remove_table_of_contents(text)
    text = remove_pdf_artifacts(text)
    text = fix_spacing(text)
    return text

In [23]:
def split_into_sentences(text):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    return sentences
def split_into_sentences(text):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    return sentences


In [24]:
def split_into_sentence_chunks(sentences, max_words=250):
    chunks = []
    current_chunk = []
    current_words = 0

    for sentence in sentences:
        word_count = len(sentence.split())

        if current_words + word_count > max_words and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_words = 0

        current_chunk.append(sentence)
        current_words += word_count

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [25]:
def build_chunk_records(chunks):
    records = []

    for index, chunk in enumerate(chunks, start=1):
        records.append({
            "chunk_id": f"chunk_{index:03d}",
            "text": chunk,
            "word_count": len(chunk.split()),
            "character_count": len(chunk)
        })

    return records

In [26]:
def build_quality_report(cleaned_text, chunks):
    word_count = len(cleaned_text.split())
    character_count = len(cleaned_text)

    if chunks:
        average_chunk_words = sum(len(chunk.split()) for chunk in chunks) / len(chunks)
    else:
        average_chunk_words = 0

    return {
        "is_empty": word_count == 0,
        "word_count": word_count,
        "character_count": character_count,
        "chunk_count": len(chunks),
        "average_chunk_words": round(average_chunk_words, 2),
        "ready_for_modeling": word_count > 50 and len(chunks) > 0
    }

In [27]:
def save_silver_output(document_id, cleaned_text, chunk_records, quality_report):
    txt_output = SILVER_FOLDER / f"{document_id}_clean.txt"
    json_output = SILVER_FOLDER / f"{document_id}_silver.json"

    txt_output.write_text(cleaned_text, encoding="utf-8")

    silver_data = {
        "document_id": document_id,
        "clean_text_file": str(txt_output),
        "quality": quality_report,
        "chunks": chunk_records,
        "processed_at": datetime.now().isoformat()
    }

    json_output.write_text(
        json.dumps(silver_data, indent=4, ensure_ascii=False),
        encoding="utf-8"
    )

In [28]:
def run_silver_layer():
    bronze_files = sorted(BRONZE_FOLDER.glob("doc_*.txt"))

    if not bronze_files:
        print("No bronze files found.")

    for file_path in bronze_files:
        document_id = file_path.stem

        print(f"Cleaning {file_path.name}...")

        raw_text = file_path.read_text(encoding="utf-8")

        cleaned_text = clean_text(raw_text)
        sentences = split_into_sentences(cleaned_text)
        chunks = split_into_sentence_chunks(sentences, max_words=250)
        chunk_records = build_chunk_records(chunks)
        quality_report = build_quality_report(cleaned_text, chunks)

        save_silver_output(
            document_id=document_id,
            cleaned_text=cleaned_text,
            chunk_records=chunk_records,
            quality_report=quality_report
        )

        print(f"Saved {document_id} to silver layer.")
        print(f"Ready for modeling: {quality_report['ready_for_modeling']}")


In [29]:

run_silver_layer()

Cleaning doc_01.txt...
Saved doc_01 to silver layer.
Ready for modeling: True
